# 05 — Phase 3: Feature Interaction Analysis

Identifies *how* language-specific features interfere with reasoning. Three competing hypotheses (per proposal §Phase 3), each with measurable predictions:

- **(a) Capacity competition** — ablating LANGUAGE features should *increase* the activation magnitude of co-located REASONING features (freed capacity).
- **(c) Attention disruption** — language ablation should *decrease* attention entropy over reasoning-relevant tokens (more focused routing).
- **(b) Circuit interference** — language features at layer L causally upstream of reasoning-feature changes at L+1, L+2, …; visualize as a directed attribution graph.

Cheapest first ((a) ≤30 min, (c) ≤45 min, (b) ≤90 min). All forward-pass-only — no generation.

## 0. Setup

In [ ]:
import os
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = 'https://github.com/kvrancic/nlp-project.git'

if IN_COLAB:
    REPO_DIR = '/content/nlp-project'
    if not Path(REPO_DIR).exists():
        get_ipython().system('git clone {REPO_URL} {REPO_DIR}')
    else:
        get_ipython().system('cd {REPO_DIR} && git pull --ff-only')
    os.chdir(REPO_DIR)
    get_ipython().system("pip install -q 'numpy>=2.0' networkx -e .")
else:
    REPO_DIR = os.getcwd()  # PI/local: assume cwd is the repo root

# Evict cached src.* so re-imports pick up the latest code
for _mod_name in [m for m in list(sys.modules) if m == 'src' or m.startswith('src.')]:
    del sys.modules[_mod_name]

if IN_COLAB:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    Path('.env').write_text(f'HF_TOKEN={os.environ["HF_TOKEN"]}\n')
else:
    from dotenv import load_dotenv
    load_dotenv()
    assert os.environ.get('HF_TOKEN'), 'HF_TOKEN missing in .env at repo root'

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_RESULTS = Path('/content/drive/MyDrive/nlp-project-results')
        DRIVE_RESULTS.mkdir(exist_ok=True, parents=True)
    except Exception:
        DRIVE_RESULTS = None
else:
    DRIVE_RESULTS = None


In [ ]:
import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from tqdm.auto import tqdm
from scipy.stats import wilcoxon
import networkx as nx

from src.config import (
    TARGET_LANGUAGES, SAE_SUBSET_LAYERS, SAE_WIDTH_16K, RESULTS_DIR, D_MODEL, MODEL_ID,
)
from src.data import load_mgsm
from src.model import load_saes_at_layers
from src.intervention import directional_ablation, get_sae_decoder_directions
from src.extraction import encode_activations_through_sae

torch.manual_seed(0); np.random.seed(0)
PRIMARY_LAYER = 17  # ablation site
DOWNSTREAM_LAYERS = [22, 29]  # where to look for reasoning-feature shifts
N_DEV = 50
N_ATTENTION = 30  # smaller because attention tensors are bulky

## 1. Load model with eager attention (needed for output_attentions)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.model import get_decoder_layers

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=os.environ['HF_TOKEN'])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# attn_implementation='eager' so output.attentions is populated. SDPA / flash
# don't expose probabilities. Eager is slower but fine for a few hundred
# forward-only passes.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto',
    token=os.environ['HF_TOKEN'], attn_implementation='eager',
)
model.eval()
DECODER_LAYERS = get_decoder_layers(model)
print(f'Model loaded with eager attention. Layers: {len(DECODER_LAYERS)} '
      f'({type(model).__name__} -> {type(model.model).__name__})')

saes = load_saes_at_layers(
    layers=SAE_SUBSET_LAYERS, width=SAE_WIDTH_16K, l0_target='medium',
)

## 2. Load Phase 1 + Phase 2b artifacts

In [ ]:
phase1 = torch.load(RESULTS_DIR / 'phase1_features.pt', weights_only=False)
phase2b = torch.load(RESULTS_DIR / 'phase2_ablation.pt', weights_only=False)

intersection = phase1['intersection_features']
top_A        = phase1['top_features_A']
reasoning_features_per_layer = phase1['reasoning_features']
confirmed_language = phase2b['confirmed_language']
best_k = phase2b['best_k']

print(f'Best k from Phase 2b: {best_k}')
print(f'Confirmed LANGUAGE features per language:')
for lang in TARGET_LANGUAGES:
    print(f'  {lang}: {confirmed_language[lang]}')
print(f'Reasoning features at layer 17: {len(reasoning_features_per_layer[17])} candidates')

In [ ]:
# Helper that mirrors Phase 2b's selection so ablation sites match.
def select_lang_features(lang, k=best_k, layer=PRIMARY_LAYER):
    out = list(confirmed_language[lang])
    for f in intersection[layer][lang]:
        if f not in out: out.append(f)
    for f in top_A[layer][lang]:
        if f not in out: out.append(f)
    return out[:k]

ablation_features = {l: select_lang_features(l) for l in TARGET_LANGUAGES}
print('Ablation feature sets per language:')
for l in TARGET_LANGUAGES:
    print(f'  {l}: {ablation_features[l]}')

mgsm = load_mgsm(TARGET_LANGUAGES)
def make_prompt(q):
    return tokenizer.apply_chat_template(
        [{'role': 'user', 'content': q}], tokenize=False, add_generation_prompt=True,
    )

## 3. Helpers — extract residuals (with optional ablation), forward-only

In [ ]:
def make_ablation_hook(directions: torch.Tensor, input_length: int):
    """Hook factory: project the span of `directions` out of the LAST INPUT TOKEN
    on the prefill pass only. Matches Phase 2b's `last`-position semantics."""
    pos = input_length - 1
    def hook(module, input, output):
        hs = output[0]
        if hs.shape[1] > pos:  # prefill
            hs[:, pos, :] = directional_ablation(hs[:, pos, :], directions)
        return (hs,) + output[1:]
    return hook

def forward_collect(prompt: str, layers_to_collect: list[int],
                    ablation_layer: int = None,
                    ablation_dirs: torch.Tensor = None,
                    want_attentions: bool = False):
    """Single forward pass on `prompt`. Returns dict with residuals at each layer
    in `layers_to_collect` (last-token only) and optionally attentions."""
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    input_len = inputs['input_ids'].shape[1]

    handles = []
    if ablation_layer is not None and ablation_dirs is not None and len(ablation_dirs) > 0:
        handles.append(DECODER_LAYERS[ablation_layer].register_forward_hook(
            make_ablation_hook(ablation_dirs.to(model.device), input_len)))
    try:
        with torch.no_grad():
            out = model(
                **inputs,
                output_hidden_states=True,
                output_attentions=want_attentions,
                use_cache=False,
            )
    finally:
        for h in handles: h.remove()

    # hidden_states[0] = embeddings, hidden_states[i+1] = output of layer i
    resid = {layer: out.hidden_states[layer + 1][:, -1, :].cpu().float()
             for layer in layers_to_collect}
    attns = None
    if want_attentions:
        # tuple of (batch, n_heads, seq, seq) per layer — keep CPU float32 to save VRAM
        attns = [a.detach().cpu().float() for a in out.attentions]
    return {'resid': resid, 'attns': attns, 'input_len': input_len}

## 4. Experiment (a) — Capacity competition

For each (lang, problem in dev), forward twice (clean / ablated at layer 17), then SAE-encode the layer-17 residual to get reasoning-feature activations. Compare distributions.

In [ ]:
REASONING_FEATS_L17 = reasoning_features_per_layer[17]
if len(REASONING_FEATS_L17) == 0:
    print('WARNING: no reasoning candidates at layer 17. Falling back to top-50 most stable.')
    # Pick features whose mean activation across languages on the same problem is
    # most consistent (lowest cross-lingual variance, highest mean).
    REASONING_FEATS_L17 = list(range(50))  # fallback placeholder; user reviews
print(f'Tracking {len(REASONING_FEATS_L17)} reasoning features at layer 17')

n_examples = N_DEV  # per language
clean_feats   = {lang: [] for lang in TARGET_LANGUAGES}  # (n, n_reason_feats)
ablated_feats = {lang: [] for lang in TARGET_LANGUAGES}

sae17 = saes[17]
for lang in TARGET_LANGUAGES:
    dirs = get_sae_decoder_directions(sae17, ablation_features[lang]).to(model.device)
    print(f'\n=== {lang}: ablating {len(ablation_features[lang])} features at L17 ===')
    for i in tqdm(range(n_examples), desc=f'{lang}'):
        prompt = make_prompt(mgsm[lang][i]['question'])
        clean = forward_collect(prompt, [17], ablation_layer=None)
        ablt  = forward_collect(prompt, [17], ablation_layer=17, ablation_dirs=dirs)
        # Encode through SAE → feature activations at this token
        clean_f = encode_activations_through_sae(clean['resid'][17].unsqueeze(0), sae17)[0]
        ablt_f  = encode_activations_through_sae(ablt['resid'][17].unsqueeze(0),  sae17)[0]
        clean_feats[lang].append(clean_f[REASONING_FEATS_L17])
        ablated_feats[lang].append(ablt_f[REASONING_FEATS_L17])
    clean_feats[lang]   = torch.stack(clean_feats[lang])
    ablated_feats[lang] = torch.stack(ablated_feats[lang])
    print(f'  shapes: clean {tuple(clean_feats[lang].shape)}, ablated {tuple(ablated_feats[lang].shape)}')
    torch.save({'clean': clean_feats, 'ablated': ablated_feats},
               RESULTS_DIR / 'phase3a_capacity_partial.pt')

In [ ]:
# Statistical test per (lang, reasoning feature): paired Wilcoxon over the
# 50 dev problems. Then aggregate effect sizes.
capacity_summary = {}
for lang in TARGET_LANGUAGES:
    rows = []
    cf = clean_feats[lang].numpy()    # (n, n_reason_feats)
    af = ablated_feats[lang].numpy()
    for j, feat_idx in enumerate(REASONING_FEATS_L17):
        diff = af[:, j] - cf[:, j]
        if np.allclose(diff, 0):
            p = 1.0; stat = 0.0
        else:
            try:
                stat, p = wilcoxon(af[:, j], cf[:, j])
            except ValueError:
                stat, p = 0.0, 1.0
        rows.append({
            'feature': int(feat_idx),
            'mean_clean': float(cf[:, j].mean()),
            'mean_ablated': float(af[:, j].mean()),
            'mean_delta': float(diff.mean()),
            'p_value': float(p),
        })
    df = pd.DataFrame(rows)
    capacity_summary[lang] = df
    n_up   = int((df['mean_delta'] > 0).sum())
    n_down = int((df['mean_delta'] < 0).sum())
    n_sig  = int((df['p_value'] < 0.05).sum())
    print(f'  {lang}: {n_up} ↑, {n_down} ↓, {n_sig} significant (p<0.05) of {len(df)}')

In [ ]:
FIG_DIR = Path('results/figures'); FIG_DIR.mkdir(exist_ok=True, parents=True)
sns.set_theme(style='whitegrid', context='paper')

# (a) Figure: distribution of mean_delta across reasoning features per language
fig, ax = plt.subplots(figsize=(7, 3.5))
rows = []
for lang in TARGET_LANGUAGES:
    for d in capacity_summary[lang]['mean_delta']:
        rows.append({'lang': lang, 'mean_delta': d})
sns.violinplot(data=pd.DataFrame(rows), x='lang', y='mean_delta', ax=ax,
               inner='quartile', density_norm='width')
ax.axhline(0, color='black', lw=0.5)
ax.set_ylabel('Δ activation (ablated − clean) of reasoning features')
ax.set_title('Phase 3 (a) — capacity competition at layer 17')
plt.tight_layout(); plt.savefig(FIG_DIR / 'fig11_capacity_competition.png', dpi=150); plt.show()

## 5. Experiment (c) — Attention disruption

For 30 dev problems per language, capture attention probabilities at layers {9, 17, 22, 29}, with and without language ablation at layer 17. Compute per-head entropy at the **last input token**'s query (matches Zhao's intervention site).

In [ ]:
ATTN_LAYERS = SAE_SUBSET_LAYERS  # {9, 17, 22, 29}

def attention_entropy_last_query(attn_layer: torch.Tensor) -> torch.Tensor:
    """Given attention probs (1, H, T, T), return per-head entropy at q=T-1."""
    # Attention probs over keys for the last query position
    p = attn_layer[0, :, -1, :].clamp_min(1e-12)  # (H, T)
    return -(p * p.log()).sum(dim=-1)             # (H,)

entropy_clean   = {layer: {l: [] for l in TARGET_LANGUAGES} for layer in ATTN_LAYERS}
entropy_ablated = {layer: {l: [] for l in TARGET_LANGUAGES} for layer in ATTN_LAYERS}

sae17 = saes[17]
for lang in TARGET_LANGUAGES:
    dirs = get_sae_decoder_directions(sae17, ablation_features[lang]).to(model.device)
    for i in tqdm(range(N_ATTENTION), desc=f'attn {lang}'):
        prompt = make_prompt(mgsm[lang][i]['question'])
        clean = forward_collect(prompt, [], ablation_layer=None, want_attentions=True)
        ablt  = forward_collect(prompt, [], ablation_layer=17,
                                ablation_dirs=dirs, want_attentions=True)
        for layer in ATTN_LAYERS:
            entropy_clean[layer][lang].append(attention_entropy_last_query(clean['attns'][layer]))
            entropy_ablated[layer][lang].append(attention_entropy_last_query(ablt['attns'][layer]))
    for layer in ATTN_LAYERS:
        entropy_clean[layer][lang]   = torch.stack(entropy_clean[layer][lang])
        entropy_ablated[layer][lang] = torch.stack(entropy_ablated[layer][lang])
    torch.save({'clean': entropy_clean, 'ablated': entropy_ablated},
               RESULTS_DIR / 'phase3c_attention_partial.pt')

In [ ]:
# Aggregate: mean entropy delta per (layer, language), averaged across heads & problems
attention_summary = []
for layer in ATTN_LAYERS:
    for lang in TARGET_LANGUAGES:
        c = entropy_clean[layer][lang]   # (n, H)
        a = entropy_ablated[layer][lang] # (n, H)
        delta = (a - c).mean().item()
        try:
            stat, p = wilcoxon(a.flatten().numpy(), c.flatten().numpy())
        except ValueError:
            stat, p = 0.0, 1.0
        attention_summary.append({
            'layer': layer, 'lang': lang, 'mean_delta_entropy': delta,
            'p_value': float(p),
        })
attn_df = pd.DataFrame(attention_summary)
print(attn_df.pivot(index='layer', columns='lang', values='mean_delta_entropy').round(4))

In [ ]:
# (c) Figure: heatmap of mean entropy delta
heat = attn_df.pivot(index='layer', columns='lang', values='mean_delta_entropy')
fig, ax = plt.subplots(figsize=(5.5, 3.2))
sns.heatmap(heat, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            cbar_kws={'label': 'Δ attention entropy (ablated − clean)'}, ax=ax)
ax.set_title('Phase 3 (c) — attention entropy shift at last input query')
plt.tight_layout(); plt.savefig(FIG_DIR / 'fig12_attention_entropy.png', dpi=150); plt.show()

## 6. Experiment (b) — Circuit interference attribution

For each (lang, dev problem in first 10), measure the change in **all SAE features** at downstream layers (22, 29) caused by ablating LANGUAGE features at layer 17. Build a directed attribution graph: edge from (L=17, lang_feat) to (L_down, feat) weighted by |Δ activation|.

In [ ]:
N_CIRCUIT = 10  # problems per language
TOP_EDGE = 30   # keep top-30 strongest downstream effects per problem

circuit_results = {lang: [] for lang in TARGET_LANGUAGES}

for lang in TARGET_LANGUAGES:
    dirs = get_sae_decoder_directions(sae17, ablation_features[lang]).to(model.device)
    print(f'\n=== circuit attribution: {lang} ===')
    for i in tqdm(range(N_CIRCUIT), desc=f'circuit {lang}'):
        prompt = make_prompt(mgsm[lang][i]['question'])
        clean = forward_collect(prompt, DOWNSTREAM_LAYERS, ablation_layer=None)
        ablt  = forward_collect(prompt, DOWNSTREAM_LAYERS, ablation_layer=17, ablation_dirs=dirs)

        per_layer_edges = {}
        for layer in DOWNSTREAM_LAYERS:
            sae_d = saes[layer]
            f_clean = encode_activations_through_sae(clean['resid'][layer].unsqueeze(0), sae_d)[0]
            f_ablt  = encode_activations_through_sae(ablt['resid'][layer].unsqueeze(0),  sae_d)[0]
            delta = (f_ablt - f_clean).abs()
            top = torch.topk(delta, k=TOP_EDGE)
            per_layer_edges[layer] = [
                {'feature': int(top.indices[j]),
                 'delta': float(f_ablt[top.indices[j]] - f_clean[top.indices[j]]),
                 'abs_delta': float(top.values[j])}
                for j in range(TOP_EDGE)
            ]
        circuit_results[lang].append({
            'problem_idx': i, 'edges': per_layer_edges,
            'ablated_features_l17': ablation_features[lang],
        })
    torch.save(circuit_results, RESULTS_DIR / 'phase3b_circuit_partial.pt')

In [ ]:
# Build aggregate attribution graph: nodes = (layer, feature), edges weighted
# by mean |Δ| over all (lang, problem) instances.
from collections import defaultdict
edge_strength = defaultdict(list)
for lang in TARGET_LANGUAGES:
    for record in circuit_results[lang]:
        src_features = record['ablated_features_l17']  # one ablation set per problem
        for downstream_layer, edges in record['edges'].items():
            for e in edges:
                # We don't know which src feature is responsible (joint ablation),
                # so the edge is from the SET to the downstream feature. Treat the
                # ablation set as a single 'super-source'.
                key = (lang, downstream_layer, e['feature'])
                edge_strength[key].append(e['abs_delta'])

summary_rows = []
for (lang, layer, feat), vs in edge_strength.items():
    summary_rows.append({
        'lang': lang, 'downstream_layer': layer, 'downstream_feature': feat,
        'mean_abs_delta': float(np.mean(vs)), 'n_problems': len(vs),
    })
edge_df = pd.DataFrame(summary_rows).sort_values('mean_abs_delta', ascending=False)
print('Top 20 strongest downstream effects (across all langs/problems):')
print(edge_df.head(20).to_string(index=False))

In [ ]:
# (b) Figure: top-N edges visualised as a bipartite graph (lang src → downstream features)
TOP_DRAW = 25
G = nx.DiGraph()
lang_color = {l: c for l, c in zip(TARGET_LANGUAGES, plt.cm.tab10.colors)}
for _, row in edge_df.head(TOP_DRAW).iterrows():
    src_node = f'{row["lang"]}-L17-set'
    dst_node = f'L{int(row["downstream_layer"])}-{int(row["downstream_feature"])}'
    G.add_edge(src_node, dst_node, weight=row['mean_abs_delta'], lang=row['lang'])

pos = nx.bipartite_layout(G,
    nodes=[n for n in G.nodes if 'L17-set' in n], align='vertical')
fig, ax = plt.subplots(figsize=(8, 6))
edge_widths = [G[u][v]['weight'] * 2 + 0.3 for u, v in G.edges]
edge_colors = [lang_color[G[u][v]['lang']] for u, v in G.edges]
nx.draw_networkx_nodes(G, pos,
    nodelist=[n for n in G.nodes if 'L17-set' in n],
    node_color=[lang_color[n.split('-')[0]] for n in G.nodes if 'L17-set' in n],
    node_size=600, ax=ax)
nx.draw_networkx_nodes(G, pos,
    nodelist=[n for n in G.nodes if 'L17-set' not in n],
    node_color='lightgrey', node_size=300, ax=ax)
nx.draw_networkx_edges(G, pos, width=edge_widths, edge_color=edge_colors,
                       arrows=True, alpha=0.7, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=7, ax=ax)
ax.set_title(f'Phase 3 (b) — top-{TOP_DRAW} downstream effects of L17 LANGUAGE ablation')
ax.axis('off')
plt.tight_layout(); plt.savefig(FIG_DIR / 'fig13_circuit_attribution.png', dpi=150); plt.show()

## 7. Save Phase 3 results

In [ ]:
phase3_payload = {
    'config': {
        'primary_layer': PRIMARY_LAYER,
        'downstream_layers': DOWNSTREAM_LAYERS,
        'attention_layers': ATTN_LAYERS,
        'n_dev': N_DEV, 'n_attention': N_ATTENTION, 'n_circuit': N_CIRCUIT,
        'best_k': best_k,
        'reasoning_features_l17': REASONING_FEATS_L17,
    },
    'capacity': {
        'clean_feats':   {l: t for l, t in clean_feats.items()},
        'ablated_feats': {l: t for l, t in ablated_feats.items()},
        'summary': {l: capacity_summary[l].to_dict('records') for l in TARGET_LANGUAGES},
    },
    'attention': {
        'entropy_clean':   {layer: dict(entropy_clean[layer])   for layer in ATTN_LAYERS},
        'entropy_ablated': {layer: dict(entropy_ablated[layer]) for layer in ATTN_LAYERS},
        'summary': attn_df.to_dict('records'),
    },
    'circuit': {
        'per_problem': circuit_results,
        'edges': edge_df.to_dict('records'),
    },
}
out = RESULTS_DIR / 'phase3_interaction.pt'
torch.save(phase3_payload, out)
print(f'Saved {out} ({out.stat().st_size/1e6:.1f} MB)')
if DRIVE_RESULTS is not None:
    torch.save(phase3_payload, DRIVE_RESULTS / 'phase3_interaction.pt')

print('\n=== Phase 3 summary ===')
print('(a) Capacity competition: per-language reasoning-feature direction shift')
for lang in TARGET_LANGUAGES:
    df = capacity_summary[lang]
    print(f'  {lang}: mean_delta = {df["mean_delta"].mean():+.4f}, '
          f'{(df["p_value"] < 0.05).sum()} sig of {len(df)}')
print('\n(c) Attention disruption: mean entropy delta per (layer, lang)')
print(attn_df.pivot(index='layer', columns='lang', values='mean_delta_entropy').round(4))
print(f'\n(b) Circuit attribution: {len(edge_df)} unique downstream edges captured.')